# Zalo AI Challenge 2026 — LifeMate (Kaggle GPU Notebook)

**Thiết lập bắt buộc trên Kaggle (thanh bên phải -> Session options):**
1. **Accelerator:** chọn **GPU T4 x2** hoặc **GPU P100** *(notebook tự khóa `CUDA_VISIBLE_DEVICES=0` để tránh lỗi `DataParallel` với 4-bit `device_map="auto"`)*.
2. **Internet:** bật **On** (để `git clone` repo và tải base model `<5B` từ HuggingFace).
3. **Add Input:** đính kèm Kaggle Dataset chứa `train` + `public_test` (hoặc file `train+public_test.zip`).

**Luồng chạy:**
- **Cell 1–2:** Clone/pull `duylhm/ZaloAIC-Lifemate`, cài thư viện, tự dò `DATA_ROOT` trong `/kaggle/input`, tạo validation splits.
- **Cell 3–4 *(Tùy chọn)*:** Benchmark ASR trên 50 file `.wav` của `train` & đo mốc Zero-shot.
- **Cell 5 *(Khuyên chạy)*:** Fine-tune QLoRA trên `train_ids` (`1,303`) & chấm Unseen-App CV (`142` unseen + `100` in-domain).
- **Cell 6–7 *(Bắt buộc)*:** Fine-tune Full (`1,545` mẫu) → Infer `public_test` (`634`) → Validate & Tải `/kaggle/working/submission.jsonl`.

In [ ]:
import os, pathlib

# Khóa 1 GPU trên Kaggle T4 x2 để Trainer + bitsandbytes 4-bit không bị xung đột DataParallel
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
%env CUDA_VISIBLE_DEVICES=0
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_DIR = pathlib.Path("/kaggle/working/ZaloAIC-Lifemate")
if not (REPO_DIR / ".git").exists():
    !rm -rf /kaggle/working/ZaloAIC-Lifemate
    !git clone https://github.com/duylhm/ZaloAIC-Lifemate.git /kaggle/working/ZaloAIC-Lifemate
else:
    !git -C /kaggle/working/ZaloAIC-Lifemate pull --ff-only

%cd /kaggle/working/ZaloAIC-Lifemate
!pip install -q -U -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Tự động tìm thư mục dữ liệu trong /kaggle/input hoặc giải nén .zip nếu có
INPUT_ROOT = pathlib.Path("/kaggle/input")
WORK_DATA = pathlib.Path("/kaggle/working/data")

def find_data_root():
    for p in INPUT_ROOT.rglob("train.jsonl"):
        cand = p.parent.parent
        if (cand / "public_test" / "public_test.jsonl").is_file():
            return cand
    for p in WORK_DATA.rglob("train.jsonl"):
        cand = p.parent.parent
        if (cand / "public_test" / "public_test.jsonl").is_file():
            return cand
    return None

data_root = find_data_root()
if data_root is None:
    zips = list(INPUT_ROOT.rglob("*.zip"))
    if zips:
        WORK_DATA.mkdir(parents=True, exist_ok=True)
        print(f"[Unzip] {zips[0]} -> {WORK_DATA}")
        !unzip -q -o "{zips[0]}" -d /kaggle/working/data
        data_root = find_data_root()

assert data_root is not None, (
    "Không tìm thấy train/train.jsonl + public_test/public_test.jsonl trong /kaggle/input. "
    "Hãy bấm '+ Add Input' và gắn dataset dữ liệu cuộc thi."
)
DATA_ROOT = str(data_root)
os.environ["LIFEMATE_DATA_ROOT"] = DATA_ROOT
print(f"[Data Ready] DATA_ROOT={DATA_ROOT}")
!python scripts/prepare_splits.py --root "{DATA_ROOT}"

In [ ]:
# Chọn base model <5B params: "Qwen/Qwen3-4B" (tắt thinking) hoặc "Qwen/Qwen2.5-3B-Instruct"
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_TAG = MODEL_ID.split("/")[-1]
CKPT_ROOT = f"/kaggle/working/checkpoints/{MODEL_TAG}"
print(f"Active MODEL_ID={MODEL_ID} | Checkpoints={CKPT_ROOT} | DATA_ROOT={DATA_ROOT}")

In [ ]:
# [Tùy chọn] Đo CER/WER tiếng Việt trên 50 file .wav tập train
!python scripts/benchmark_asr.py --root "{DATA_ROOT}" --models vinai/PhoWhisper-large openai/whisper-large-v3-turbo --limit 50 --num-beams 3

In [ ]:
# [Tùy chọn] Đo mốc Zero-shot trên 242 mẫu val_all trước khi fine-tune
!python run_inference.py \
  --root "{DATA_ROOT}" \
  --split train \
  --ids-file splits/val_all.json \
  --model "{MODEL_ID}" \
  --force \
  --output /kaggle/working/val_zeroshot.jsonl \
  --eval

In [ ]:
# Fine-tune QLoRA trên train_ids (1,303) & chấm điểm Unseen-App CV (142 unseen + 100 in-domain)
VAL_ADAPTER = f"{CKPT_ROOT}_val"
!python train_sft.py \
  --root "{DATA_ROOT}" \
  --model "{MODEL_ID}" \
  --split-mode val \
  --epochs 2 \
  --aug-copies 1 \
  --max-seq-len 6144 \
  --output-dir "{VAL_ADAPTER}"

!python run_inference.py \
  --root "{DATA_ROOT}" \
  --split train \
  --ids-file splits/val_all.json \
  --model "{MODEL_ID}" \
  --adapter "{VAL_ADAPTER}" \
  --force \
  --output /kaggle/working/val_sft.jsonl \
  --eval

!python scripts/inspect_errors.py \
  --root "{DATA_ROOT}" \
  --split train \
  --ids-file splits/val_all.json \
  --submission /kaggle/working/val_sft.jsonl \
  --max-per-bucket 5

In [ ]:
# Fine-tune Full 1,545 mẫu -> Infer public_test (634) -> Validate & xuất /kaggle/working/submission.jsonl
FULL_ADAPTER = f"{CKPT_ROOT}_full"
!python train_sft.py \
  --root "{DATA_ROOT}" \
  --model "{MODEL_ID}" \
  --split-mode full \
  --epochs 2 \
  --aug-copies 1 \
  --max-seq-len 6144 \
  --output-dir "{FULL_ADAPTER}"

!python run_inference.py \
  --root "{DATA_ROOT}" \
  --split public_test \
  --model "{MODEL_ID}" \
  --adapter "{FULL_ADAPTER}" \
  --force \
  --output /kaggle/working/submission.jsonl

In [ ]:
# Kiểm tra file kết quả & tạo link tải trực tiếp trên Kaggle
import json, pathlib, shutil
from IPython.display import FileLink, display

sub_path = pathlib.Path("/kaggle/working/submission.jsonl")
assert sub_path.is_file(), "Chưa có /kaggle/working/submission.jsonl. Hãy chạy Cell 6 trước."

lines = [l for l in sub_path.read_text(encoding="utf-8").splitlines() if l.strip()]
print(f"[OK] {sub_path} | Số dòng: {len(lines)}/634")
print("Dòng đầu:", lines[0][:160])

# Link tải trực tiếp (tương đối từ /kaggle/working/ZaloAIC-Lifemate nếu đang ở trong repo)
local_copy = pathlib.Path("submission.jsonl")
if local_copy.resolve() != sub_path.resolve():
    shutil.copy2(sub_path, local_copy)
display(FileLink("submission.jsonl"))